In [1]:
import time
import os
import pandas as pd
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont
from moviepy import ImageClip

from edge_tts import list_voices, Communicate
from moviepy import AudioFileClip
from pydub import AudioSegment
from pydub.exceptions import CouldntDecodeError

pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

# 0. Settings

In [14]:
data_settings = {
    'category': '交通标志',
    'output_path_base': 'output/anomia_shorts/'
}
data_settings['output_path'] = os.path.join(
    data_settings['output_path_base'],
    data_settings['category'].replace(' ', '_')
)
data_settings['output_path_audio'] = os.path.join(
    data_settings['output_path'],
    'audio_files'
)

# Get category index, pinyin, english from data
df_categories = pd.read_csv('static/texts/lists/anomia_categories.csv')
this_category = df_categories[df_categories['chinese'] == data_settings['category']].iloc[0]
data_settings['category_pinyin'] = this_category['pinyin']
data_settings['category_english'] = this_category['english']
data_settings['category_id'] = this_category['id']
data_settings

{'category': '交通标志',
 'output_path_base': 'output/anomia_shorts/',
 'output_path': 'output/anomia_shorts/交通标志',
 'output_path_audio': 'output/anomia_shorts/交通标志/audio_files',
 'category_pinyin': 'jiāo tōng biāo zhì',
 'category_english': 'traffic signs',
 'category_id': np.int64(1)}

In [15]:
audio_settings = {
    'voice_name_zh': 'zh-CN-XiaoxiaoNeural',
    'audio_plan': 'ctitle_c2word',
    'pause_ms_beginning': 150,
    'pause_ms_within_word': 200,
    'pause_ms_between': 500,
}
audio_settings

{'voice_name_zh': 'zh-CN-XiaoxiaoNeural',
 'audio_plan': 'ctitle_c2word',
 'pause_ms_beginning': 150,
 'pause_ms_within_word': 200,
 'pause_ms_between': 500}

In [16]:
BG_SIZE = (720, 1280)
video_settings = {
    'bg_size': BG_SIZE,
    'bg_color': 'white',
    'text_color': 'black',

    'max_line_length_buffer_size': 60,
    'decrease_font_step_size': 1,
    'font_path': '/System/Library/Fonts/STHeiti Medium.ttc',

    'title_settings': {
        'x': 30,
        'y': 30,
        'spacing': 30,
        'align': 'center',
        'font_size': 50,
        'fill': {'chinese': '#FFFFFF', 'pinyin': '#CCCCCC', 'english': '#CCCCCC'},
    },

    'words_settings': {
        'x': {'chinese': '30', 'pinyin': '200', 'english': '400'},
        'y': 200,
        'spacing': 30,
        'font_size': 36,
        'align': {'chinese': 'left', 'pinyin': 'left', 'english': 'left'},
        'fill': {'chinese': '#FFFFFF', 'pinyin': '#CCCCCC', 'english': '#CCCCCC'},
    },

    'horizontal_line': {
        'x': 10,
        'y': 140,
        'color': "#1E90FF",
        'width': 4,
    },

    'logo': {
        'font_name': 'Arial Black',
        'font_size': 20,
        'x': 30,
        'y': 30,
        'color1': "#3E78D6",
        'color2': "#2FDDFC",
    },

    'category_index': {
        'index_value': 1,
        'index_total': 100,
        'font_name': 'Arial Black',
        'font_size': 36,
        'x': BG_SIZE[0] - 30 - 100,
        'y': 30,
        'color1': "#FFFFFF",
        'color2': "#CCCCCC",
    },
}
video_settings

{'bg_size': (720, 1280),
 'bg_color': 'white',
 'text_color': 'black',
 'max_line_length_buffer_size': 60,
 'decrease_font_step_size': 1,
 'font_path': '/System/Library/Fonts/STHeiti Medium.ttc',
 'title_settings': {'x': 30,
  'y': 30,
  'spacing': 30,
  'align': 'center',
  'font_size': 50,
  'fill': {'chinese': '#FFFFFF', 'pinyin': '#CCCCCC', 'english': '#CCCCCC'}},
 'words_settings': {'x': {'chinese': '30', 'pinyin': '200', 'english': '400'},
  'y': 200,
  'spacing': 30,
  'font_size': 36,
  'align': {'chinese': 'left', 'pinyin': 'left', 'english': 'left'},
  'fill': {'chinese': '#FFFFFF', 'pinyin': '#CCCCCC', 'english': '#CCCCCC'}},
 'horizontal_line': {'x': 10, 'y': 140, 'color': '#1E90FF', 'width': 4},
 'logo': {'font_name': 'Arial Black',
  'font_size': 20,
  'x': 30,
  'y': 30,
  'color1': '#3E78D6',
  'color2': '#2FDDFC'},
 'category_index': {'index_value': 1,
  'index_total': 100,
  'font_name': 'Arial Black',
  'font_size': 36,
  'x': 590,
  'y': 30,
  'color1': '#FFFFFF',
 

In [3]:
if not os.path.exists(data_settings['output_path_base']):
    os.mkdir(data_settings['output_path_base'])
if not os.path.exists(data_settings['output_path']):
    os.mkdir(data_settings['output_path'])
if not os.path.exists(data_settings['output_path_audio']):
    os.mkdir(data_settings['output_path_audio'])

{'category': '交通标志',
 'output_path_base': 'output/anomia_shorts/',
 'output_path': 'output/anomia_shorts/交通标志',
 'output_path_audio': 'output/anomia_shorts/交通标志/audio_files',
 'category_pinyin': 'jiāo tōng biāo zhì',
 'category_english': 'traffic signs',
 'category_id': np.int64(1)}

# 1. Load data

In [ ]:
df_raw = pd.read_csv('static/texts/lists/anomia_examples.csv')
print(len(df_raw))
df_raw.head()

847


,id,category,chinese,pinyin,english
0,1,交通标志,停车标志,tíng chē biāo zhì,stop sign
1,2,交通标志,限速标志,xiàn sù biāo zhì,speed limit sign
2,3,交通标志,禁止通行,jìn zhǐ tōng xíng,no entry
3,4,交通标志,人行横道,rén xíng héng dào,pedestrian crossing
4,5,交通标志,注意行人,zhù yì xíng rén,watch for pedestrians


In [5]:
# Print for creating audio
df_filt = df_raw[df_raw['category'] == data_settings['category']].reset_index(drop=True)
example_words = df_filt['chinese'].values
example_words

array(['停车标志', '限速标志', '禁止通行', '人行横道', '注意行人', '红绿灯', '单行道', '让行', '学校区域',
       '禁止鸣笛'], dtype=object)

# 2. Generate spoken audio

In [6]:
# import pandas as pd
# from collections import defaultdict
# import os
# import time
# import numpy as np
# import shutil
# import datetime

# from edge_tts import list_voices, Communicate

# voice_name = 'zh-CN-XiaoxiaoNeural'
# category = '交通标志'
# examples = ['停车标志', '限速标志', '禁止通行', '人行横道', '注意行人', '红绿灯', '单行道', '让行', '学校区域',
#        '禁止鸣笛']

# if not os.path.exists(category):
#     os.mkdir(category)


# def single_tts_call(text, voice_name, output_file_name):
#   communicate = Communicate(text, voice_name)
#   communicate.save_sync(output_file_name)

# start_time = time.time()
# for text_str in [category] + examples:
#   print(time.time()-start_time, text_str)
#   single_tts_call(text_str, voice_name, f'{category}/{text_str}.mp3')

# current_datetime = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# shutil.make_archive(f"{category}_{current_datetime}", 'zip', category)


# 3. Stitch audio together to final audio

In [ ]:
dict_audio_durations = defaultdict(list)
if audio_settings['audio_plan'] == 'ctitle_c2word':
    current_start_time = 0

    # beginning pause
    pause_beginning = AudioSegment.silent(duration=audio_settings['pause_ms_beginning'])
    combined = pause_beginning
    dict_audio_durations['audio_path'].append('pause_beginning')
    dict_audio_durations['duration'].append(audio_settings['pause_ms_beginning'] / 1000)
    dict_audio_durations['start_time'].append(current_start_time)
    current_start_time += audio_settings['pause_ms_beginning'] / 1000
    dict_audio_durations['end_time'].append(current_start_time)

    # title
    title_audio_path = f"{data_settings['output_path_audio']}/{data_settings['category']}.mp3"
    audio = AudioSegment.from_mp3(title_audio_path)
    combined += audio
    dict_audio_durations['audio_path'].append(title_audio_path)
    dict_audio_durations['duration'].append(audio.duration_seconds)
    dict_audio_durations['start_time'].append(current_start_time)
    current_start_time += audio.duration_seconds
    dict_audio_durations['end_time'].append(current_start_time)

    # words
    for i_word, word in enumerate(example_words):
        # inter-word pause
        pause_inter_word = AudioSegment.silent(duration=audio_settings['pause_ms_between'])
        combined += pause_inter_word
        dict_audio_durations['audio_path'].append('inter_word_pause')
        dict_audio_durations['duration'].append(audio_settings['pause_ms_between'] / 1000)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio_settings['pause_ms_between'] / 1000
        dict_audio_durations['end_time'].append(current_start_time)
        
        # word audio
        word_audio_path = f"{data_settings['output_path_audio']}/{word}.mp3"
        audio = AudioSegment.from_mp3(word_audio_path)
        combined += audio 
        dict_audio_durations['audio_path'].append(word_audio_path)
        dict_audio_durations['duration'].append(audio.duration_seconds)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio.duration_seconds
        dict_audio_durations['end_time'].append(current_start_time)

        # within-word pause
        pause_within_word = AudioSegment.silent(duration=audio_settings['pause_ms_within_word'])
        combined += pause_within_word
        dict_audio_durations['audio_path'].append('within_word_pause')
        dict_audio_durations['duration'].append(audio_settings['pause_ms_within_word'] / 1000)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio_settings['pause_ms_within_word'] / 1000
        dict_audio_durations['end_time'].append(current_start_time)

        # word again audio
        combined += audio 
        dict_audio_durations['audio_path'].append(word_audio_path)
        dict_audio_durations['duration'].append(audio.duration_seconds)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio.duration_seconds
        dict_audio_durations['end_time'].append(current_start_time)

# export the combined audio file
combined.export(f"{data_settings['output_path_audio']}/combined.mp3", format="mp3")

# Add in static slide audio into dataframe of audio durations
df_durations = pd.DataFrame(dict_audio_durations)
df_durations.head(10)


,audio_path,duration,start_time,end_time
0,pause_beginning,0.150,0.000,0.150
1,output/anomia_shorts/交通标志/audio_files/交通标志.mp3,1.680,0.150,1.830
2,inter_word_pause,0.500,1.830,2.330
3,output/anomia_shorts/交通标志/audio_files/停车标志.mp3,1.656,2.330,3.986
4,within_word_pause,0.200,3.986,4.186
5,output/anomia_shorts/交通标志/audio_files/停车标志.mp3,1.656,4.186,5.842
6,inter_word_pause,0.500,5.842,6.342
7,output/anomia_shorts/交通标志/audio_files/限速标志.mp3,1.704,6.342,8.046
8,within_word_pause,0.200,8.046,8.246
9,output/anomia_shorts/交通标志/audio_files/限速标志.mp3,1.704,8.246,9.950


In [8]:
df_durations.tail(10)

,audio_path,duration,start_time,end_time
32,within_word_pause,0.200,30.678,30.878
33,output/anomia_shorts/交通标志/audio_files/让行.mp3,1.368,30.878,32.246
34,inter_word_pause,0.500,32.246,32.746
35,output/anomia_shorts/交通标志/audio_files/学校区域.mp3,1.680,32.746,34.426
36,within_word_pause,0.200,34.426,34.626
37,output/anomia_shorts/交通标志/audio_files/学校区域.mp3,1.680,34.626,36.306
38,inter_word_pause,0.500,36.306,36.806
39,output/anomia_shorts/交通标志/audio_files/禁止鸣笛.mp3,1.608,36.806,38.414
40,within_word_pause,0.200,38.414,38.614
41,output/anomia_shorts/交通标志/audio_files/禁止鸣笛.mp3,1.608,38.614,40.222


# 4. Create images

TODO


# 5. Stitch images together for video

TODO

# 6. Output video with audio

TODO